# Experiment 21

Notebook ini membaca artefak hasil `python -m src.experiment_21.run_experiment_21`. Jalankan pipeline Python terlebih dahulu, lalu jalankan semua cell notebook ini untuk inspeksi hasil.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results' / 'experiment_21'
required = [
    RESULTS / 'experiment_21_summary.json',
    RESULTS / 'experiment_21_table.csv',
    RESULTS / 'front_history_exp21.json',
    RESULTS / 'side_history_exp21.json',
    RESULTS / 'fusion_predictions_exp21.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Artefak belum lengkap. Jalankan: python -m src.experiment_21.run_experiment_21\n' + '\n'.join(missing))

summary = json.loads((RESULTS / 'experiment_21_summary.json').read_text(encoding='utf-8'))
table = pd.read_csv(RESULTS / 'experiment_21_table.csv')
front_history = json.loads((RESULTS / 'front_history_exp21.json').read_text(encoding='utf-8'))
side_history = json.loads((RESULTS / 'side_history_exp21.json').read_text(encoding='utf-8'))
fusion_predictions = pd.read_csv(RESULTS / 'fusion_predictions_exp21.csv')
table

## Ringkasan Protokol

In [ ]:
print(summary['description'])
print('\nFinal v1 reference:', summary['reference_final_v1'])
print('Exp 8 reference:', summary['reference_exp8'])

## Kurva Training

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.plot(epochs, history['train_loss'], label='train loss')
    ax1.plot(epochs, history['val_loss'], label='val loss')
    ax1.set_xlabel('epoch')
    ax1.set_ylabel('loss')
    ax2 = ax1.twinx()
    ax2.plot(epochs, history['val_macro_f1'], color='tab:green', label='val Macro F1')
    ax2.set_ylabel('Macro F1')
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='lower right')
    ax1.set_title(title)
    ax1.grid(alpha=0.25)
    plt.show()

plot_history(front_history, 'Experiment 21 Front')
plot_history(side_history, 'Experiment 21 Side')

## Perbandingan Metrik

In [ ]:
display_cols = ['method', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'safe_to_phone', 'phone_to_safe', 'generalization_status']
table[display_cols]

In [ ]:
ax = table.set_index('method')['f1_macro'].plot(kind='bar', figsize=(8, 4), color=['#3b6ea8', '#879957', '#c05a3b', '#7b5fa8'])
ax.axhline(summary['reference_final_v1']['average_fusion_f1_macro'], color='black', linestyle='--', linewidth=1, label='Final v1 fusion F1')
ax.set_ylabel('Macro F1')
ax.set_title('Experiment 21 Macro F1')
ax.legend()
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Sampel Prediksi Fusion

In [ ]:
fusion_predictions.head(10)